In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from category_encoders import CountEncoder # pip install category_encoders

# For DBSCAN + UMAP
import umap # pip install umap-learn
from sklearn.cluster import DBSCAN
from sklearn.neighbors import KNeighborsClassifier

# For Autoencoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Cell 2: Load Data and Prevent Leakage
# Load the dataset
df = pd.read_csv(r'data\submissions.csv')

# Separate target immediately to prevent any accidental leakage during feature engineering
X = df.drop(columns=['is_valid'])
y = df['is_valid'].astype(int) # Assuming 0 = fake/anomalous, 1 = genuine

# Perform train-test split BEFORE any preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

In [ ]:
# Cell 3: Custom Preprocessing Function
def engineer_features(data):
    df_engineered = data.copy()
    
    # 1. Monetary Normalization
    df_engineered['exchangeRate'] = df_engineered['exchangeRate'].fillna(1.0)
    
    monetary_cols = [
        'baseSalary', 'totalCompensation', 'firstYearTotalCompensation', 
        'avgAnnualStockGrantValue', 'totalStockGrantValue', 'avgAnnualBonusValue'
    ]
    for col in monetary_cols:
        df_engineered[col] = pd.to_numeric(df_engineered[col], errors='coerce').fillna(0)
        df_engineered[f'{col}_normalized'] = df_engineered[col] * df_engineered['exchangeRate']
    
    # 2. Date Handling
    df_engineered['offerDate'] = pd.to_datetime(df_engineered['offerDate'], errors='coerce', utc=True)
    df_engineered['submittedAt'] = pd.to_datetime(df_engineered['submittedAt'], errors='coerce', utc=True)
    df_engineered['days_to_submit'] = (df_engineered['submittedAt'] - df_engineered['offerDate']).dt.days
    
    # 3. Numeric Conversions for behavioral features
    df_engineered['userAccountAgeDays'] = pd.to_numeric(df_engineered['userAccountAgeDays'], errors='coerce')
    df_engineered['userPriorSubmissions'] = pd.to_numeric(df_engineered['userPriorSubmissions'], errors='coerce')
    
    # 4. Drop useless, redundant, and complex non-numeric columns
    cols_to_drop = monetary_cols + [
        'uuid', 'offerDate', 'submittedAt', 'exchangeRate', 'baseSalaryCurrency', 
        'stockGrantCurrency', 'bonusCurrency', 'userCurrency', 'userExchangeRate',
        'companyInfo_icon', 'companyInfo_name', 'companyInfo_slug', 'companyInfo_registered',
        'tags', 'otherDetails', 'userId', 'threadId', 'commentsAndRepliesCount',
        'vestingSchedule', 'additionalBonuses' # Dropping complex string/JSON structures
    ]
    df_engineered.drop(columns=[col for col in cols_to_drop if col in df_engineered.columns], inplace=True)
    
    return df_engineered

X_train_eng = engineer_features(X_train)
X_test_eng = engineer_features(X_test)

In [ ]:
# Cell 4: Encoders and Scalers Pipeline

# High-cardinality categorical features (including focusTag)
high_cardinality_cols = [
    'company', 'title', 'jobFamily', 'jobFamilySlug', 'focusTag',
    'location', 'locationSlug', 'ipHash', 'deviceHash', 
    'cityId', 'dmaId', 'countryId'
]
# Retain only those present in dataset
high_cardinality_cols = [col for col in high_cardinality_cols if col in X_train_eng.columns]

# Low-cardinality categorical features
low_cardinality_cols = [
    'level', 'workArrangement', 'compPerspective', 'salaryFormat', 
    'employmentType', 'gender', 'ethnicity', 'education', 
    'stockType', 'companySize', 'fundingStage'
]
low_cardinality_cols = [col for col in low_cardinality_cols if col in X_train_eng.columns]

# Force all remaining object/string columns to string type for safety
categorical_cols = high_cardinality_cols + low_cardinality_cols
for col in categorical_cols:
    X_train_eng[col] = X_train_eng[col].astype(str)
    X_test_eng[col] = X_test_eng[col].astype(str)

# Strictly filter numeric columns using select_dtypes to prevent string contamination
numeric_cols = X_train_eng.select_dtypes(include=[np.number]).columns.tolist()

# 1. Pipeline for Numerical features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# 2. Pipeline for High Cardinality
high_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', CountEncoder(combine_min_nan_groups=True))
])

# 3. Pipeline for Low Cardinality
low_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('high_card', high_card_transformer, high_cardinality_cols),
        ('low_card', low_card_transformer, low_cardinality_cols)
    ]
)

# Fit strictly on train, transform both
X_train_processed = preprocessor.fit_transform(X_train_eng)
X_test_processed = preprocessor.transform(X_test_eng)

# Convert outputs to float32
X_train_processed = X_train_processed.astype(np.float32)
X_test_processed = X_test_processed.astype(np.float32)

print(f"Final feature matrix shape: {X_train_processed.shape}")

In [ ]:
# Cell 5 (Autoencoder): Define and Train Model
input_dim = X_train_processed.shape[1]

# Define Autoencoder Architecture
input_layer = Input(shape=(input_dim,))
# Encoder
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dropout(0.2)(encoded)
encoded = Dense(64, activation='relu')(encoded)
encoded = Dense(32, activation='relu')(encoded) # Bottleneck

# Decoder
decoded = Dense(64, activation='relu')(encoded)
decoded = Dropout(0.2)(decoded)
decoded = Dense(128, activation='relu')(decoded)
output_layer = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

print(autoencoder.summary())

# Train the model. 
# In classic anomaly detection, we train only on "normal" data to learn its pattern.
# We will isolate normal data (is_valid == 1) strictly from the TRAIN set to prevent leakage.
X_train_normal = X_train_processed[y_train == 1]

history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

# Plot Training Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Autoencoder Loss')
plt.legend()
plt.show()

In [ ]:
# Cell 6 (Autoencoder): Evaluation & Thresholding
# Calculate reconstruction errors (MSE)

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

train_predictions = autoencoder.predict(X_train_processed)
train_mse = np.mean(np.power(X_train_processed - train_predictions, 2), axis=1)

test_predictions = autoencoder.predict(X_test_processed)
test_mse = np.mean(np.power(X_test_processed - test_predictions, 2), axis=1)

# Define a threshold based on the training data
# Top 5% highest reconstruction errors are considered anomalies
threshold = np.percentile(train_mse, 80)
print(f"Reconstruction Error Threshold: {threshold:.4f}")

# Predictions: 0 = Fake/Anomaly, 1 = Valid
y_train_pred_ae = np.where(train_mse > threshold, 0, 1)
y_test_pred_ae = np.where(test_mse > threshold, 0, 1)

# Training Set Evaluation
print("\n--- Training Set Evaluation (Autoencoder) ---")
print(f"Accuracy: {accuracy_score(y_train, y_train_pred_ae):.4f}")
print(f"F1 Score: {f1_score(y_train, y_train_pred_ae, pos_label=0, zero_division=0):.4f}")
print("\nClassification Report:")
print(classification_report(
    y_train,
    y_train_pred_ae,
    target_names=['Fake (0)', 'Valid (1)'],
    zero_division=0
))

# Test Set Evaluation
print("\n--- Test Set Evaluation (Autoencoder) ---")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred_ae):.4f}")
print(f"F1 Score: {f1_score(y_test, y_test_pred_ae, pos_label=0, zero_division=0):.4f}")
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred_ae,
    target_names=['Fake (0)', 'Valid (1)'],
    zero_division=0
))